# C# — Le bilan « mathocalypse » — quand l'IA publie 372 théorèmes en un soir

**Série Complexity — accrétion 07c** · Public : **Recherche** · Hommage fondateur : **Scott Aaronson** (« The Mathocalypse », Shtetl-Optimized, 7/10/2026, mise à jour 12/10/2026)

**Navigation** : [<< Complexity-07b — Le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) | [Index de la série](README.md)

Le 6 octobre 2026, OpenAI publie d'un coup **372 résultats de recherche** obtenus par ses agents, sur recommandation de son groupe consultatif AGMAI (Timothy Gowers, Edward Witten et autres mathématiciens). Ce carnet décrypte cette vague pour l'étudiant : ce qui a été prouvé, ce que ça change, ce que ça ne change pas — et ce que « comprendre une preuve » veut dire quand plus personne ne la comprend.

**Distinction de périmètre, écrite ici comme exigé** : ce carnet fait **comprendre** les résultats (décryptage, Zoo, mesures numériques) ; la **distillation formelle** relève de l'EPIC #19898, dont l'arbitrage du pli 1 (09/10/2026) classe ces résultats précis en **non-distillation écrite** — motif : aucun lac du dépôt ne porte la théorie correspondante — et confie la matière **en veille à cette série-ci** : ce carnet est ce premier dépôt, il renvoie, il ne refait pas. Conventions de la série : chaque fait est **Mesuré** (sortie de ce carnet) ou **Cité** (source nominative). Source primaire archivée au gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\AI and Mathematics - the October 2026 Release and its Reception\2026 - Aaronson - blog post 10169 sur la release openai-math (scottaaronson.blog, OA).html`.

**Jumeau C#.** Les calculs et les bancs 3SUM sont natifs. Le Python versionné est la source primaire consultée ; les grands résultats et citations sont rapportés de son billet cité, sans nouvelle preuve vérifiée ici. Les générateurs et les chronos diffèrent entre langages. [Source Python](Complexity-07c-LeBilanMathocalypse-Python.ipynb).


## 1. La mécanique de la vague — ce qui a tourné, combien de temps

Le récit factuel du billet : des agents GPT-Pro ont reçu environ 4 000 problèmes ouverts de longue date ; environ 10 % ont abouti, donnant les 372 résultats publiés. Le coût moyen : **environ 3 heures de calcul de niveau GPT-Pro par problème résolu**. Une correction honnête vit dans le fil : une première version du billet annonçait 8 000 problèmes et ~5 % de succès, corrigée par les commentateurs (la source officielle, `github.com/openai/math`, dit ~4 000 problèmes posés) — même les témoins directs se trompent dans l'urgence, et le corriger en public est la moitié de l'épistémologie de ce carnet.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Diagnostics;
using System.Globalization;

// Le point décimal est invariant dans les mesures affichées.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
// Les hypothèses sont celles rapportées par le Python : trois heures par problème résolu.
int poses = 4000, resolus = 372, heuresParResolu = 3;
double taux = (double)resolus / poses;
int heures = resolus * heuresParResolu;
Console.WriteLine($"{resolus}/{poses} = {taux:P1} ; {heures} heures pour les seuls résolus, soit {heures / 24.0:F1} jours.");
Console.WriteLine("Première estimation 8000/~5 %, corrigée en 4000 par la source rapportée.");


372/4000 = 9.3 % ; 1116 heures pour les seuls résolus, soit 46.5 jours.


Première estimation 8000/~5 %, corrigée en 4000 par la source rapportée.


## 2. Unique Games — la conjecture qui est tombée, et la mathématicienne « cooked »

La pièce maîtresse de la vague : une preuve de la **conjecture Unique Games** de Subhash Khot (UGC, 2002) — l'un des grands problèmes d'optimisation, qui gouverne la dureté d'approximation de Max Cut et de toutes les CSP. Le billet raconte que le fils de la complexity theorist **Dana Moshkovitz** lui a lancé : *« mommy, I heard you got cooked! »* — elle travaillait vers cette conjecture depuis des années ; le billet publie d'ailleurs sa réaction à chaud (commentaire #5 : elle doute que le « teleport to the peak » suffise à comprendre la montagne). La preuve, elle, « invente un code complètement nouveau et bizarre avec un test de bruit », une récursion folle — et les preuves d'inapproximabilité des applications (Max Cut, CSP) contournent désormais l'UGC elles-mêmes (Cité — billet, §résultats).

## 3. Derandomization et quantique — L=BPL, parité ∉ QAC$^0$, area law

Trois questions de trente ans et plus, tombées dans la même fournée : **L=BPL** (le logspace probabiliste égale le logspace déterministe — l'une des grandes conjectures de dérandomization) ; **la parité n'est pas dans QAC$^0$** (question ouverte de complexité quantique depuis 1999, sur laquelle « many of my colleagues had been closing in on ») ; et la **loi d'aire pour Hamiltoniens gapés 2D**, l'un des problèmes ouverts majeurs de la complexité hamiltonienne. Une seule des trois a sa case au [Complexity Zoo du 03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) : **L** — la carte du 03c s'arrête à $L \subseteq NL \subseteq P \subseteq NP \subseteq PSPACE$ (avec EXPTIME et coNP) et ne porte **ni BPL ni QAC$^0$**. Le Zoo reste la carte de lecture de la hiérarchie que la vague traverse ; BPL et QAC$^0$ restent à lire ailleurs.

In [2]:
// Les statuts viennent du billet cité, sans vérification nouvelle des preuves.
var zoo = new Dictionary<string, (string avant, string apres, string role)>
{
    ["UGC (Khot 2002)"] = ("conjecture ouverte", "prouvée par la vague", "approximation Max Cut/CSP"),
    ["L = BPL"] = ("conjecture ouverte", "prouvé", "dérandomisation logspace"),
    ["parité hors QAC^0"] = ("ouverte depuis 1999", "prouvé", "circuits quantiques"),
    ["loi d'aire 2D gappée"] = ("problème majeur ouvert", "prouvé", "complexité hamiltonienne"),
    ["P vs NP"] = ("ouvert", "toujours ouvert", "absent de la liste")
};
foreach (var(nom, v)in zoo)
    Console.WriteLine($"{nom, -28} {v.avant} -> {v.apres} ; {v.role}");
Console.WriteLine($"{zoo.Count(v => v.Value.avant != v.Value.apres && v.Value.apres != "toujours ouvert")}/{zoo.Count} entrées modifiées");


UGC (Khot 2002)              conjecture ouverte -> prouvée par la vague ; approximation Max Cut/CSP


L = BPL                      conjecture ouverte -> prouvé ; dérandomisation logspace


parité hors QAC^0            ouverte depuis 1999 -> prouvé ; circuits quantiques


loi d'aire 2D gappée         problème majeur ouvert -> prouvé ; complexité hamiltonienne


P vs NP                      ouvert -> toujours ouvert ; absent de la liste


4/5 entrées modifiées


## Exercice 1 — étendre le mini-zoo

La table de la section 3 est un dictionnaire. Écrire `EtendreZoo(zoo, nouvelles)` qui ajoute les nouvelles entrées **sans écraser** les existantes (une clé déjà présente ne se remplace pas — elle signale) et retourne le zoo étendu.

In [3]:
Dictionary<string, (string avant, string apres, string role)> EtendreZoo(Dictionary<string, (string avant, string apres, string role)> initial, Dictionary<string, (string avant, string apres, string role)> nouvelles)
{
    // TODO : créer un nouveau dictionnaire et ne jamais écraser les clés existantes.
    return null;
}

var avantZoo = new Dictionary<string, (string avant, string apres, string role)>(zoo);
var nouvelles = new Dictionary<string, (string, string, string)>
{
    ["P=BPP"] = ("ouvert", "toujours ouvert", "absent"),
    ["L = BPL"] = ("faux", "faux", "faux")
};
var essaiZoo = EtendreZoo(zoo, nouvelles);
var essaiVide = EtendreZoo(zoo, new());
if (essaiZoo is null && essaiVide is null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    if (zoo.Count != avantZoo.Count || avantZoo.Any(kv => !zoo.TryGetValue(kv.Key, out var present) || present != kv.Value))
        throw new Exception("Zoo initial modifié");
    if (essaiZoo is null || essaiVide is null || object.ReferenceEquals(essaiZoo, zoo) || object.ReferenceEquals(essaiVide, zoo) || essaiZoo.Count != 6 || essaiVide.Count != 5 || !essaiZoo.TryGetValue("P=BPP", out var ajout) || ajout != nouvelles["P=BPP"])
        throw new Exception("Extension incorrecte");
    foreach (var(cle, valeur)in zoo)
        if (!essaiZoo.TryGetValue(cle, out var presente) || presente != valeur || !essaiVide.TryGetValue(cle, out var presenteVide) || presenteVide != valeur)
            throw new Exception("Entrée existante modifiée");
    Console.WriteLine("Copie à six entrées, cinq originales conservées");
}


Exercice 1 à compléter


## 4. Les barrières algorithmiques — un exposant rationnel, et treize 9

Deux résultats cassent des **barrières d'exposant** vieilles de décennies : le **produit matriciel en $n^{9/4+o(1)}$** — un exposant *rationnel* pour une fois, obtenu par une approche « complètement différente » de la lignée $O(n^{2.373})$ ; et la **multiplication d'entiers en $O(n \log^{0.9999999999999} n)$** — en dessous de la barrière $n \log n$ qui tenait depuis les années 1960 (Knuth-Schönhage-Strassen). Deux subtilités que le fil du billet rend vivantes : le résultat matriciel **n'est pas constructif** (commentaire de Bram Cohen : une existence de type Ramsey, pas un algorithme qu'on peut exécuter) ; et l'exposant $0.9999999999999$ — **treize neuf** — dit une victoire asymptotique totalement invisible en pratique. On mesure les deux :

In [4]:
// Comparer les puissances et un surcoût hypothétique, sans affirmer un temps réel.
foreach (double n in new[]
{
    1e6,
    1e9,
    1e12
}

)
    Console.WriteLine($"matmul n={n:E0} : n^0.123={Math.Pow(n, 0.123):F2}×");
double exposantBascule = Math.Log10(100) / (2.373 - 2.25);
Console.WriteLine($"Surcoût 100× : bascule vers 10^{exposantBascule:F2}");
foreach (double bits in new[]
{
    64.0,
    1e6
}

)
    Console.WriteLine($"bits={bits:G} : ancien/nouveau={bits / Math.Pow(bits, 0.9999999999999):F15}");


matmul n=1E+006 : n^0.123=5.47×


matmul n=1E+009 : n^0.123=12.79×


matmul n=1E+012 : n^0.123=29.92×


Surcoût 100× : bascule vers 10^16.26


bits=64 : ancien/nouveau=1.000000000000416


bits=1000000 : ancien/nouveau=1.000000000001382


## Exercice 2 — la bascule d'un crossover d'exposant

Section 4 : le nouvel algorithme en $n^{2.25}$ ne gagne que si sa constante n'est pas trop mauvaise. Écrire `Bascule(ancien, nouveau, cAncien, cNouveau)` : le plus petit $n$ (puissance de 2, énumérée) tel que $c_{nouveau} \cdot n^{e_{nouveau}} \leq c_{ancien} \cdot n^{e_{ancien}}$, ou `null` si l'énumération dépasse $2^{64}$.
Le retour est un `BigInteger?` : $2^{64}$ fait partie du domaine et dépasse `ulong.MaxValue`.


In [5]:
BigInteger? Bascule(double ancien, double nouveau, double cAncien, double cNouveau)
{
    // TODO : chercher les puissances de 2 depuis 2^0 jusqu'à 2^64 inclus.
    return null;
}

var essaiBascule = Bascule(2.373, 2.25, 1, 1);
var grand = Bascule(1, 0, 1, Math.Pow(2, 64));
if (essaiBascule is null && grand is null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    var cas = new (double a, double b, double ca, double cb, BigInteger? attendu)[]
    {
        (2.373, 2.25, 1, 1, BigInteger.One),
        (2.373, 2.25, 1, 100, BigInteger.One << 55),
        (1, 0, 1, Math.Pow(2, 64), BigInteger.One << 64),
        (0, 1, 1, 2, null)
    };
    foreach (var c in cas)
    {
        var r = Bascule(c.a, c.b, c.ca, c.cb);
        if (r != c.attendu)
            throw new Exception("Bascule incorrecte ou partielle");
        Console.WriteLine($"Bascule : {(r?.ToString() ?? "absente")}");
    }
}


Exercice 2 à compléter


## 5. 3SUM et SETH — le contrepoint humain de la veille

Le **lundi 5 octobre au soir**, Virginia Williams et Josh Alman ont posté sur arXiv un préprint qui résout **3SUM en $O(n^{1.9992})$** et **All-Pairs Shortest Paths en $O(n^{2.9995})$** — réfutant des conjectures vieux de cinquante ans selon lesquelles les bonnes bornes étaient $n^2$ et $n^3$. Deux leçons dans le billet : (a) **l'idée cruciale a été fournie par un modèle Anthropic**, pas OpenAI — mais Anthropic a choisi une voie de diffusion opposée : plutôt que publier les solutions non digérées au monde, il les a transmises à des mathématiciens humains (Williams et Alman) qui les ont travaillées, vérifiées et signées ; (b) **SETH reste intacte** : le papier « dit explicitement que SETH est unaffected and still open » — 3SUM sous-quadratique ne tue pas la Strong Exponential Time Hypothesis. On mesure ce que $n^2 \to n^{1.9992}$ vaut vraiment, et le scaling réel d'un 3SUM quadratique :

In [6]:
// Chaque complément trouvé doit posséder un indice distinct des deux indices de la paire.
bool TroisSommeHash(int[] liste)
{
    var indices = new Dictionary<long, List<int>>();
    for (int k = 0; k < liste.Length; k++)
    {
        if (!indices.TryGetValue(liste[k], out var pos))
            indices[liste[k]] = pos = new List<int>();
        pos.Add(k);
    }

    for (int i = 0; i < liste.Length; i++)
        for (int j = i + 1; j < liste.Length; j++)
            if (indices.TryGetValue(-((long)liste[i] + liste[j]), out var pos))
                foreach (int k in pos)
                    if (k != i && k != j)
                        return true;
    return false;
}

if (TroisSommeHash(new[] { -2, 1, 4 }) || !TroisSommeHash(new[] { 5, -3, -2, 40 }) || TroisSommeHash(new[] { int.MaxValue, int.MaxValue, 2 }) || !TroisSommeHash(new[] { int.MaxValue, int.MinValue, 1 }))
    throw new Exception("Indices 3SUM incorrects");
Console.WriteLine("Garde indices distincts : négatif et positif conformes.");
foreach (double n in new[]
{
    1e6,
    1e12
}

)
    Console.WriteLine($"3SUM n={n:E0} : facteur n^0.0008={Math.Pow(n, 0.0008):F4}");
var rng = new Random(42);
var mesures = new List<(int n, double secondes)>();
foreach (int n in new[]
{
    400,
    800,
    1600,
    3200
}

)
{
    // Des valeurs positives garantissent le parcours complet des paires.
    var a = Enumerable.Range(0, n).Select(_ => rng.Next(1, 1_000_001)).ToArray();
    var chrono = Stopwatch.StartNew();
    bool r = TroisSommeHash(a);
    chrono.Stop();
    if (r)
        throw new Exception("Un tableau positif ne contient aucun triplet nul");
    mesures.Add((n, chrono.Elapsed.TotalSeconds));
    Console.WriteLine($"n={n}, {chrono.Elapsed.TotalMilliseconds:F2} ms, résultat={r}");
}

double exposantEmp = Math.Log(mesures[^1].secondes / mesures[1].secondes) / Math.Log((double)mesures[^1].n / mesures[1].n);
Console.WriteLine($"Exposant empirique mesuré 800→3200 : {exposantEmp:F2}");


Garde indices distincts : négatif et positif conformes.


3SUM n=1E+006 : facteur n^0.0008=1.0111


3SUM n=1E+012 : facteur n^0.0008=1.0224


n=400, 2.03 ms, résultat=False


n=800, 7.05 ms, résultat=False


n=1600, 34.11 ms, résultat=False


n=3200, 132.60 ms, résultat=False


Exposant empirique mesuré 800→3200 : 2.12


## Exercice 3 — un 3SUM sous-quadratique ? Non : mesurer le quadratique

La cellule de la section 5 mesure le scaling du 3SUM par hachage. Écrire `TroisSommeNaif(L)` (trois boucles imbriquées, $O(n^3)$, même réponse booléenne), puis mesurer les deux implémentations sur $n = 500$ et $n = 1000$ (seed 42) et rapporter le rapport des temps.
Additionner les valeurs en `long` avant comparaison : trois `int` peuvent déborder leur type même lorsque la somme mathématique est définie.


In [7]:
bool? TroisSommeNaif(int[] a)
{
    // TODO : trois boucles i<j<k, sans réutiliser un indice ; élargir avant addition.
    return null;
}

var positif = TroisSommeNaif(new[] { 5, -3, -2, 40 });
var negatif = TroisSommeNaif(new[] { -2, 1, 4 });
if (positif is null && negatif is null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    if (positif != true || negatif != false || TroisSommeNaif(new[] { 0, 0, 0 }) != true || TroisSommeNaif(new[] { 0, 0 }) != false || TroisSommeNaif(new[] { int.MaxValue, int.MaxValue, 2 }) != false || TroisSommeNaif(new[] { int.MaxValue, int.MinValue, 1 }) != true)
        throw new Exception("Indices ou résultat 3SUM incorrects");
    var r = new Random(42);
    foreach (int n in new[]
    {
        500,
        1000
    }

    )
    {
        // Les valeurs positives forcent les deux moteurs à parcourir tout le banc.
        var a = Enumerable.Range(0, n).Select(_ => r.Next(1, 1_000_001)).ToArray();
        var sw = Stopwatch.StartNew();
        var naive = TroisSommeNaif(a);
        sw.Stop();
        double tn = sw.Elapsed.TotalSeconds;
        sw.Restart();
        bool hash = TroisSommeHash(a);
        sw.Stop();
        double th = sw.Elapsed.TotalSeconds;
        if (naive is null || naive.Value != hash)
            throw new Exception("Naïf et hachage divergent");
        Console.WriteLine($"n={n}: naïf={tn:F4}s, hachage={th:F4}s, ratio={tn / th:F2}");
    }
}


Exercice 3 à compléter


## 6. Ce qui manque de la liste — la ligne de crête intacte

Le point le plus important pour un cours de complexité : **ce qui n'est PAS dans les 372**. Le billet : *« P≠NP isn't there, nor even P=BPP or NEXP⊄P/poly, and surely not for lack of trying. »* Les plus grands problèmes ouverts de la théorie de la complexité — P vs NP, P=BPP, NEXP⊄P/poly — n'ont pas bougé. Côté Millennium, le billet relève des **progrès partiels** vers Riemann, Hodge (rétracté pour une erreur de signe) et BSD — « the majority of the remaining Millennium Problems » — mais aucune résolution complète. La vague a cassé des conjectures *circulables* (UGC, L=BPL, des exposants) là où des agents pouvaient explorer des espaces de preuve massifs ; elle n'a rien fait aux questions structurelles qui résistent depuis un demi-siècle. « Plateau ou pas plateau » n'est pas la question : la carte de ce qui est tombé et de ce qui tient est la réponse (Cité — billet, §ce qui manque).

## 7. Épistémologie — preuves aliénées, certificats Lean, deux modèles de diffusion

Trois fils se nouent :

- **Preuves aliénées** : pour plusieurs des 372 résultats, « il semble qu'aucun humain n'ait compris la moindre de ces preuves ». Il existe un **certificat Lean** pour certaines d'entre elles — *pas toutes* : la vérification formelle progresse en parallèle de la compréhension, elle ne la remplace pas.
- **AGMAI** : le groupe consultatif (Gowers, Witten…) a publié une déclaration « très soigneusement phrasée, n'endossant ni ne condamnant » ce qu'OpenAI a fait — et le fil révèle qu'AGMAI avait demandé à OpenAI de détailler ressources, nombre de prompts, problèmes tentés et échoués.
- **Deux modèles de diffusion** : OpenAI publie les solutions brutes au monde entier ; Anthropic remet les idées à des mathématiciens choisis qui les digèrent et signent. Le billet pèse les deux : la première accélère tout le monde mais noie la communauté ; la seconde produit des preuves comprises mais « met une entreprise privée en position de choisir les émissaires de l'IA ». Le débat est ouvert — c'est aussi une question de **politique scientifique**, pas seulement de mathématiques (Cité — billet + commentaires #5, #42, #59).

## Gap Mathlib

À la date de rédaction (14/10/2026), aucun de ces résultats (UGC, L=BPL, parité ∉ QAC$^0$, exposants matmul/entiers, 3SUM sous-quadratique) n'a de formalisation référencée dans ce dépôt — les certificats Lean mentionnés par le billet vivent côté OpenAI, pas ici. *(Renvoi du 28/10/2026 : le volet 3SUM est désormais couvert — voir [03e — Barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb), qui lit la formalisation anthropics/formal-math · 3sum-apsp publiée par les auteurs ; les autres volets de la liste restent sans formalisation référencée.)* Arbitrage rendu : le pli 1 de l'EPIC **#19898** (09/10/2026) classe ces résultats en **non-distillation écrite** — motif : un pli exige un socle de lac à confronter, et aucun lac du dépôt ne porte la théorie correspondante ; la matière est consignée **en veille pour la série Complexity elle-même**, ce carnet en est le premier dépôt. Pour l'indécidabilité et Gödel, l'ancre reste [#15066] (livrée, lake `formal_logic_lean`).

**État au 2026-11-23.** La vague a d'ailleurs trouvé ses formalisations, sur d'autres théories, dans la série [ANALYSE](../../SymbolicAI/Lean/Analyse/README.md) — sensibilité (06), percolation (07), Ramsey (08), tuilage (09), synthèse transverse (11), comptage de matching (12). Les cinq volets de ce carnet (UGC, L=BPL, parité, exposants, 3SUM) y restent sans formalisation référencée, sauf le 3SUM lu en [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) : le gap tient.


## Conclusion — la description courte d'un théorème lourd

La vague d'octobre 2026, lue depuis ce dépôt, tient dans le miroir déjà posé par le [07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : des agents ont produit des **descriptions courtes** (des preuves de quelques pages, parfois un certificat Lean) d'**énoncés lourds** (UGC, L=BPL, des barrières d'exposants vieille de cinquante ans) — la face « preuves » du tour des grands nombres du [07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb), et leur pendant côté données — le MDL de la série Compression ([Compression-06](../Compression/Compression-06-MDL-Kolmogorov-Python.ipynb)) ; l'inégalité de Kraft elle-même est dans [Compression-01](../Compression/Compression-01-ShannonFano-Prefixe-Python.ipynb) et son jumeau Lean [Compression-01b — Kraft-McMillan](../Compression/Compression_lean/Compression-01b-Kraft-McMillan-Lean.ipynb) : ce qui est rare n'est pas d'écrire court, c'est de **vérifier** ce qui est écrit court. La ligne de crête — P vs NP — n'a pas bougé ; ce qui a changé, c'est le coût de production d'une preuve circulable, et donc la question épistémique : une preuve que personne ne comprend mais qu'un vérificateur formel valide est-elle une compréhension ? (#19898 a arbitré ces résultats non-distillables sur nos lacs à ce jour — la veille vit dans cette série, en prose mesurée.) Le Zoo ([03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb)) reste la carte où lire la hiérarchie que la vague traverse — ni BPL ni QAC$^0$ n'y figurent.

## À retenir

1. **La vague tient en trois nombres mesurés.** 372 résultats publiés sur environ 4000 problèmes posés : taux de résolution 9.3 %, compute de 1 116 h GPT-Pro pour les seuls problèmes résolus — environ 46 jours d'un seul agent non stop.
2. **La ligne de crête n'a pas bougé.** Le mini-zoo par code rend 4 entrées bougées sur 5 — UGC, L=BPL, parité $\notin$ QAC$^0$, loi d'aire 2D gapée — et P vs NP toujours ouverte : la vague a cassé des conjectures circulables, pas les questions structurelles.
3. **Les victoires d'exposants sont exactes et invisibles.** Le produit matriciel gagne un facteur 30 à $n = 10^{12}$ (écart d'exposant $n^{0.123}$) mais ne bascule qu'à $n \sim 10^{16.3}$ si le nouveau schéma coûte 100× plus par opération ; le 3SUM sous $n^2$ ($n^{1.9992}$) ne gagne que 1.0224× à $n = 10^{12}$ ; la multiplication d'entiers sous la barrière $n \log n$ gagne 1.000000000000416× à $n = 2^{64}$.
4. **Le mur quadratique reste quadratique quand on le mesure.** Le 3SUM par hachage, chronométré de $n = 800$ à $n = 3200$, affiche un exposant empirique propre au rejeu — l'asymptotique nouvelle ne change rien à ce qui tourne à taille réaliste.
5. **La frontière Mesuré/Cité traverse tout le carnet.** Le récit (Moshkovitz « cooked », AGMAI, les deux modèles de diffusion) est **Cité** depuis le billet archivé ; aucune de ces preuves n'est formalisée dans ce dépôt (Gap Mathlib), et la correction publique 8000 → 4000 problèmes fait partie de l'épistémologie.

## Vérifiez votre compréhension

1. Pourquoi le carnet conserve-t-il la mention « une première version disait 8000 problèmes, ~5 % » alors que ces chiffres sont faux ?
2. En quoi le résultat matriciel $n^{9/4+o(1)}$ peut-il être « non constructif », et que mesure la bascule à $n \sim 10^{16.3}$ ?
3. Que signifie « SETH intacte » dans le contrepoint Williams-Alman, alors que 3SUM passe sous $n^2$ ?
4. Quelles entrées du mini-zoo ont bougé, et pourquoi celle qui manque est-elle la plus importante pour un cours de complexité ?

<details><summary>Réponses</summary>

1. Parce que la correction publique fait partie de la leçon : même un témoignage direct se trompe dans l'urgence, les commentateurs l'ont corrigé, et la source officielle (github.com/openai/math) dit environ 4000 problèmes posés. Conserver la trace de l'erreur, c'est montrer l'épistémologie en action.
2. Le billet rapporte (commentaire de Bram Cohen) que le résultat est de type existence, à la Ramsey — sans algorithme qu'on peut exécuter. La bascule à $n \sim 10^{16.3}$ suppose un surcoût constant de 100× par opération pour le nouveau schéma : elle dit à partir de quelle taille astronomique l'avantage d'exposant paierait la constante, donc pourquoi la victoire reste invisible en pratique.
3. Le papier de Williams et Alman dit explicitement que SETH est « unaffected and still open » : casser 3SUM ($O(n^{1.9992})$) et All-Pairs Shortest Paths ($O(n^{2.9995})$) réfute ces conjectures-là, pas la Strong Exponential Time Hypothesis — les barrières ne tombent pas ensemble.
4. UGC (prouvée par la vague), L=BPL, parité $\notin$ QAC$^0$, loi d'aire 2D gapée : quatre entrées sur cinq. La manquante est P vs NP, absente de la liste « pas faute d'avoir essayé » — la question structurelle centrale, qui fixe la limite de ce que l'exploration massive d'espaces de preuve obtient.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Compression-06 — MDL : décrire le modèle autant que les données](../Compression/Compression-06-MDL-Kolmogorov-Python.ipynb) — dernière accrétion de la série : le pendant MDL cité en conclusion, écrire court du côté des données cette fois.
- **Dans la même série** : [Complexity-03c — le Zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — la carte où lire L et la hiérarchie qui l'englobe ; ni BPL ni QAC$^0$ n'y figurent ; [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — l'autre lecture à la source d'une annonce d'octobre 2026, surfaces Lean comprises.
- **Côté lacs** : les formalisations de la vague d'octobre 2026 vivent dans la série [ANALYSE](../../SymbolicAI/Lean/Analyse/README.md) (06 → 12) — sensibilité, percolation, Ramsey, tuilage, synthèse transverse, comptage de matching : d'autres théories que les cinq volets de ce carnet, qui restent en veille.
- **Carnet voisin d'une autre série** : [Lean-03b — Formalized Formal Logic](../../SymbolicAI/Lean/Lean-03b-Formalized-Formal-Logic-Lean-Python.ipynb) — ce que « vérifié par une machine » veut dire, mot à mot, sur un lac du dépôt (ancre #15066 citée par le Gap Mathlib).
- **Lectures** : Scott Aaronson, « The Mathocalypse » (Shtetl-Optimized, 7/10/2026, mis à jour le 12/10/2026 — copie archivée au gisement partagé, chemin cité en tête de ce carnet) ; le dépôt github.com/openai/math, la source officielle des résultats.
